# MLflow: Tracking, Managing, and Deploying ML Models

## What Is MLflow?

Imagine you are a scientist running 100 experiments to find the best cookie recipe.  
You try different amounts of sugar, different baking temperatures, different flours.  
Without notes, you'd forget what worked. MLflow is your lab notebook — it records every experiment automatically.

**MLflow** is an open-source platform for the complete ML lifecycle:
- **Tracking**: log parameters, metrics, and artifacts from experiments
- **Models**: standard format for saving/loading models from any framework
- **Registry**: version control for models (staging → production → archived)
- **Projects**: reproducible packaging of ML code

## Why MLflow?

Without MLflow, ML teams face these problems:
- "Which model is in production? Was it the one trained on Tuesday?"
- "I got 94% accuracy last week — what hyperparameters did I use?"
- "The model on the server crashes. Which version is it?"

MLflow solves all of these by being the single source of truth for your experiments.

## Resources

- **Docs**: [https://mlflow.org/docs/latest/](https://mlflow.org/docs/latest/)
- **GitHub**: [https://github.com/mlflow/mlflow](https://github.com/mlflow/mlflow)
- **YouTube — MLflow Tutorial**: [https://www.youtube.com/watch?v=2g3qgMxE5gY](https://www.youtube.com/watch?v=2g3qgMxE5gY)
- **YouTube — MLflow Model Registry**: [https://www.youtube.com/watch?v=0JzLnkzrGEs](https://www.youtube.com/watch?v=0JzLnkzrGEs)

## Installation

```bash
pip install mlflow
pip install mlflow[extras]  # includes sklearn, pytorch, tensorflow integrations
```

MLflow works **locally without any server** — it saves runs to a local `mlruns/` folder by default.  
Run `mlflow ui` in your terminal to see the dashboard at http://localhost:5000.

In [ ]:
import os
import tempfile
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f"MLflow version: {mlflow.__version__}")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not installed — simulated output shown. Install: pip install mlflow")

# Use a temp directory so runs don't clutter the project
MLRUNS_DIR = tempfile.mkdtemp(prefix='mlflow_demo_')
if MLFLOW_AVAILABLE:
    mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")
    print(f"MLflow tracking directory: {MLRUNS_DIR}")

## Core Concept 1: Experiments and Runs

**Experiment** = a named group of related runs (e.g., "churn_prediction_v2")  
**Run** = one execution of your training script with specific hyperparameters

Think of it like: Experiment = lab project, Run = one trial within that project.

Each run can log:
- **Parameters** (`log_param`): hyperparameters you set (learning rate, batch size)
- **Metrics** (`log_metric`): things you measure (accuracy, loss) — can track over time
- **Artifacts** (`log_artifact`): files (model weights, plots, data samples)
- **Tags** (`set_tag`): free-form labels ("version:2", "author:alice")

In [ ]:
# Generate synthetic classification dataset
np.random.seed(42)
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class balance: {y.mean():.2%} positive")


# ── A single MLflow run ───────────────────────────────────────────────────────
def train_and_log(model, model_name, params, X_tr, X_te, y_tr, y_te):
    """Train a model and log everything to MLflow."""
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=model_name):
            # Log hyperparameters
            mlflow.log_params(params)
            mlflow.set_tag("model_type", model_name)
            mlflow.set_tag("dataset", "synthetic_classification")

            # Train
            model.fit(X_tr, y_tr)

            # Evaluate
            y_pred = model.predict(X_te)
            y_prob = model.predict_proba(X_te)[:, 1]
            acc    = accuracy_score(y_te, y_pred)
            auc    = roc_auc_score(y_te, y_prob)

            # Log metrics
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("roc_auc",  auc)

            # Log the model itself
            mlflow.sklearn.log_model(model, "model")

            run_id = mlflow.active_run().info.run_id
            print(f"  {model_name:30s} acc={acc:.4f}  auc={auc:.4f}  run_id={run_id[:8]}...")
            return run_id, acc, auc
    else:
        # Simulate without MLflow
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]
        acc = accuracy_score(y_te, y_pred)
        auc = roc_auc_score(y_te, y_prob)
        print(f"  {model_name:30s} acc={acc:.4f}  auc={auc:.4f}  (simulated logging)")
        return f"sim_{model_name[:8]}", acc, auc


# Set experiment
if MLFLOW_AVAILABLE:
    mlflow.set_experiment("classification_comparison")

print("Training and logging 5 models...")
print("-" * 70)

models_to_compare = [
    (LogisticRegression(max_iter=1000, C=1.0),
     "LogisticRegression", {"C": 1.0, "max_iter": 1000}),
    (DecisionTreeClassifier(max_depth=5, random_state=42),
     "DecisionTree",       {"max_depth": 5}),
    (RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
     "RandomForest",       {"n_estimators": 100, "max_depth": 8}),
    (GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
     "GradientBoosting",   {"n_estimators": 100, "learning_rate": 0.1}),
    (RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=2, random_state=42),
     "RandomForest_tuned", {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 2}),
]

results = []
for model, name, params in models_to_compare:
    run_id, acc, auc = train_and_log(model, name, params, X_train_s, X_test_s, y_train, y_test)
    results.append({"model": name, "run_id": run_id, "accuracy": acc, "roc_auc": auc})

results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
print("\nLeaderboard:")
print(results_df.to_string(index=False))

## Core Concept 2: Querying Runs Programmatically

MLflow stores all runs in a database you can query in Python.  
This lets you find the best run, compare experiments, and automate model promotion.

In [ ]:
if MLFLOW_AVAILABLE:
    # Search all runs in the experiment
    runs = mlflow.search_runs(
        experiment_names=["classification_comparison"],
        order_by=["metrics.roc_auc DESC"]
    )
    cols = ["run_id", "tags.model_type", "metrics.accuracy", "metrics.roc_auc",
            "params.n_estimators", "params.max_depth"]
    print("All logged runs (sorted by ROC-AUC):")
    print(runs[[c for c in cols if c in runs.columns]].head(10).to_string())

    # Get the best run
    best_run = runs.iloc[0]
    best_run_id = best_run["run_id"]
    print(f"\nBest run: {best_run_id[:8]}... AUC={best_run['metrics.roc_auc']:.4f}")
else:
    print("Simulated mlflow.search_runs() output:")
    print()
    sim = pd.DataFrame([
        {"run_id": "a1b2c3d4", "model_type": "RandomForest_tuned",  "accuracy": 0.8975, "roc_auc": 0.9612},
        {"run_id": "e5f6g7h8", "model_type": "GradientBoosting",    "accuracy": 0.8850, "roc_auc": 0.9534},
        {"run_id": "i9j0k1l2", "model_type": "RandomForest",        "accuracy": 0.8775, "roc_auc": 0.9489},
        {"run_id": "m3n4o5p6", "model_type": "LogisticRegression",  "accuracy": 0.8450, "roc_auc": 0.9213},
        {"run_id": "q7r8s9t0", "model_type": "DecisionTree",        "accuracy": 0.8200, "roc_auc": 0.8967},
    ])
    print(sim.to_string(index=False))
    best_run_id = "a1b2c3d4"
    print(f"\nBest run: {best_run_id}... AUC=0.9612")

## Core Concept 3: Loading a Logged Model

MLflow saves models in a framework-agnostic format called **MLmodel**.  
You can load any logged model using its run ID — no need to keep track of file paths.

In [ ]:
if MLFLOW_AVAILABLE:
    # Load the best model directly from run
    model_uri = f"runs:/{best_run_id}/model"
    loaded_model = mlflow.sklearn.load_model(model_uri)

    # Use for inference — exactly like the original model
    sample = X_test_s[:5]
    predictions = loaded_model.predict(sample)
    probabilities = loaded_model.predict_proba(sample)[:, 1]

    print(f"Loaded model from: {model_uri}")
    print(f"Predictions:   {predictions}")
    print(f"Probabilities: {probabilities.round(3)}")

    # Also: load as a generic pyfunc model (framework-agnostic)
    pyfunc_model = mlflow.pyfunc.load_model(model_uri)
    pyfunc_preds = pyfunc_model.predict(pd.DataFrame(sample))
    print(f"pyfunc predictions: {pyfunc_preds.values[:5]}")
else:
    print("Loading model from MLflow (simulated):")
    print()
    print("  model_uri = 'runs:/a1b2c3d4/model'")
    print("  loaded_model = mlflow.sklearn.load_model(model_uri)")
    print("  # loaded_model is a fully functional sklearn model")
    print("  predictions = loaded_model.predict(X_test)")
    print()
    print("  # Generic pyfunc loading (works for any ML framework)")
    print("  pyfunc_model = mlflow.pyfunc.load_model(model_uri)")
    print("  predictions = pyfunc_model.predict(pd.DataFrame(X_test))")
    print()
    print("  Simulated output:")
    print("  Predictions:   [1 0 1 1 0]")
    print("  Probabilities: [0.892 0.134 0.756 0.943 0.201]")

## Core Concept 4: MLflow autolog

`mlflow.autolog()` automatically logs everything from supported frameworks (sklearn, PyTorch, TensorFlow, XGBoost, LightGBM, etc.) with a single line.  
No manual `log_param` calls needed!

In [ ]:
if MLFLOW_AVAILABLE:
    # One-line autolog — captures ALL sklearn params/metrics automatically
    mlflow.sklearn.autolog(
        log_input_examples=True,   # save sample inputs
        log_model_signatures=True, # save input/output schema
        log_post_training_metrics=True
    )

    mlflow.set_experiment("autolog_demo")

    with mlflow.start_run(run_name="rf_autolog"):
        rf = RandomForestClassifier(n_estimators=50, random_state=42)
        rf.fit(X_train_s, y_train)  # autolog captures: n_estimators, max_depth, etc.
        # evaluate on test set — autolog captures these metrics too
        test_acc = rf.score(X_test_s, y_test)
        print(f"Autolog run complete. Test accuracy: {test_acc:.4f}")
        print("MLflow automatically captured: all RF hyperparameters, training metrics, model")

    mlflow.sklearn.autolog(disable=True)  # turn off when done
else:
    print("mlflow.autolog() example (simulated):")
    print()
    print("  mlflow.sklearn.autolog()  # ONE LINE — that's it!")
    print()                  
    print("  with mlflow.start_run():")
    print("      rf = RandomForestClassifier(n_estimators=50)")
    print("      rf.fit(X_train, y_train)")
    print("      # ^ autolog captured: n_estimators=50, max_depth=None, ...")
    print()
    print("  Automatically logged parameters:")
    print("    n_estimators: 50")
    print("    max_depth: None")
    print("    min_samples_split: 2")
    print("    ... (all 15 sklearn RF params)")
    print()
    print("  Automatically logged metrics:")
    print("    training_accuracy_score: 0.9987")
    print("    training_roc_auc_score: 0.9998")

## Core Concept 5: Model Registry

The Model Registry is version control for ML models. It tracks the lifecycle:

```
Run logged → Register → Staging → Production → Archived
                 ↓
           "model_v1"  "model_v2"  "model_v3"
```

**Why this matters**: Without a registry, teams deploy models by copying files around — error-prone.  
The registry ensures everyone uses the same model and you can roll back instantly.

In [ ]:
if MLFLOW_AVAILABLE:
    client = mlflow.tracking.MlflowClient()

    # Register the best model from our experiment
    model_uri = f"runs:/{best_run_id}/model"
    registered = mlflow.register_model(
        model_uri=model_uri,
        name="churn_classifier"
    )
    print(f"Registered: {registered.name} v{registered.version}")

    # Transition to Staging
    client.transition_model_version_stage(
        name="churn_classifier",
        version=registered.version,
        stage="Staging"
    )
    print(f"Transitioned to Staging")

    # In production code: load the current production model
    # mlflow.pyfunc.load_model("models:/churn_classifier/Production")

    # List all registered models
    for mv in client.search_model_versions("name='churn_classifier'"):
        print(f"  Version {mv.version}: stage={mv.current_stage}, run={mv.run_id[:8]}...")
else:
    print("Model Registry workflow (simulated):")
    print()
    print("  # Register a model from a run")
    print("  registered = mlflow.register_model('runs:/a1b2c3d4/model', 'churn_classifier')")
    print("  # → Version 1 created in registry")
    print()
    print("  # Promote to staging for testing")
    print("  client.transition_model_version_stage('churn_classifier', 1, 'Staging')")
    print()
    print("  # After testing: promote to production")
    print("  client.transition_model_version_stage('churn_classifier', 1, 'Production')")
    print()
    print("  # In serving code: always load latest production model")
    print("  model = mlflow.pyfunc.load_model('models:/churn_classifier/Production')")
    print()
    print("  Registered versions:")
    print("  Version 1: stage=Production, run=a1b2c3d4...")
    print("  Version 2: stage=Staging,    run=e5f6g7h8...")

## Core Concept 6: Logging Custom Metrics Over Time

You can log metrics at each training step — MLflow stores the full history.  
This gives you training curves you can visualize in the UI.

In [ ]:
import math

if MLFLOW_AVAILABLE:
    mlflow.set_experiment("training_curves")

    with mlflow.start_run(run_name="simulated_training"):
        mlflow.log_param("model", "neural_network")
        mlflow.log_param("learning_rate", 0.001)

        # Simulate epoch-by-epoch training
        for epoch in range(1, 21):
            train_loss = 2.5 * math.exp(-0.15 * epoch) + 0.05 * np.random.randn()
            val_loss   = 2.5 * math.exp(-0.12 * epoch) + 0.1  + 0.08 * np.random.randn()
            val_acc    = 1 - 0.9 * math.exp(-0.15 * epoch)

            # log_metric with step parameter creates a time series
            mlflow.log_metric("train_loss", max(0, train_loss), step=epoch)
            mlflow.log_metric("val_loss",   max(0, val_loss),   step=epoch)
            mlflow.log_metric("val_acc",    min(1, val_acc),    step=epoch)

    print("Logged 20 epochs of training metrics. View in MLflow UI: mlflow ui")
    print("Training curves are visible at http://localhost:5000")
else:
    print("Simulated epoch-by-epoch logging:")
    print()
    print("  with mlflow.start_run():")
    print("      for epoch in range(1, 21):")
    print("          mlflow.log_metric('train_loss', loss, step=epoch)")
    print("          mlflow.log_metric('val_acc',    acc,  step=epoch)")
    print()
    print("  Simulated training curve:")
    for ep in [1, 5, 10, 15, 20]:
        loss = 2.5 * math.exp(-0.15 * ep)
        acc  = 1 - 0.9 * math.exp(-0.15 * ep)
        print(f"    Epoch {ep:2d}: loss={loss:.3f}  val_acc={acc:.3f}")

## MLflow Serving

MLflow can serve any registered model as a REST API with one command:

```bash
# Serve the Production model as REST API on port 5001
mlflow models serve -m "models:/churn_classifier/Production" --port 5001

# Call it with curl
curl -X POST http://localhost:5001/invocations \
  -H 'Content-Type: application/json' \
  -d '{"dataframe_records": [{"feature_0": 1.2, "feature_1": -0.5, ...}]}'
```

Or build a Docker container:
```bash
mlflow models build-docker -m "models:/churn_classifier/Production" -n churn-api
docker run -p 5001:8080 churn-api
```

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Nested runs confusion | Runs appear under wrong experiment | Always set experiment before `start_run()` |
| Not closing runs | Runs stuck as "RUNNING" | Use `with mlflow.start_run():` (context manager) |
| Wrong tracking URI | Runs saved to wrong place | Call `mlflow.set_tracking_uri()` at script start |
| Log metric before start_run | `ActiveRunError` | Ensure run is active first |
| Logging large artifacts every epoch | Slow training, huge storage | Log model only at end, not every epoch |
| autolog in Jupyter re-logging | Duplicate runs | Call `mlflow.end_run()` or restart kernel |
| Model registry on local filesystem | Won't work for team sharing | Use MLflow server with database backend for teams |

## Mini Project: Model Competition — Find the Best Regressor

We'll use MLflow to run a systematic model comparison for a regression task,  
automatically register the winner, and show how to load it for production.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Synthetic regression dataset — predict house prices
X_reg, y_reg = make_regression(n_samples=3000, n_features=15, noise=30, random_state=42)
y_reg = np.abs(y_reg) * 1000 + 50000  # scale to realistic house prices

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
X_tr_s = scaler2.fit_transform(X_tr)
X_te_s  = scaler2.transform(X_te)

regressors = [
    (Ridge(alpha=1.0),                                          "Ridge",       {"alpha": 1.0}),
    (Lasso(alpha=0.1),                                          "Lasso",       {"alpha": 0.1}),
    (ElasticNet(alpha=0.1, l1_ratio=0.5),                       "ElasticNet",  {"alpha": 0.1, "l1_ratio": 0.5}),
    (RandomForestRegressor(n_estimators=100, random_state=42),  "RandomForest",{"n_estimators": 100}),
    (GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
     "GBR", {"n_estimators": 100, "lr": 0.1}),
]

if MLFLOW_AVAILABLE:
    mlflow.set_experiment("house_price_regression")

print("Regression Model Competition")
print("=" * 60)

best_rmse = float('inf')
best_reg_run_id = None

for model, name, params in regressors:
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=name):
            mlflow.log_params(params)
            model.fit(X_tr_s, y_tr)
            y_pred = model.predict(X_te_s)
            rmse = np.sqrt(mean_squared_error(y_te, y_pred))
            mae  = mean_absolute_error(y_te, y_pred)
            r2   = r2_score(y_te, y_pred)
            mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
            mlflow.sklearn.log_model(model, "model")
            run_id = mlflow.active_run().info.run_id
            if rmse < best_rmse:
                best_rmse = rmse
                best_reg_run_id = run_id
    else:
        model.fit(X_tr_s, y_tr)
        y_pred = model.predict(X_te_s)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        mae  = mean_absolute_error(y_te, y_pred)
        r2   = r2_score(y_te, y_pred)
        run_id = f"sim_{name[:6]}"
        if rmse < best_rmse:
            best_rmse = rmse
            best_reg_run_id = run_id

    print(f"  {name:20s}  RMSE={rmse:8.1f}  MAE={mae:8.1f}  R²={r2:.4f}")

print(f"\nBest model run: {best_reg_run_id} (RMSE={best_rmse:.1f})")

if MLFLOW_AVAILABLE:
    # Auto-register winner
    winner_uri = f"runs:/{best_reg_run_id}/model"
    mlflow.register_model(winner_uri, "house_price_model")
    print("Winner registered as 'house_price_model' in Model Registry!")

## Interview Questions and Answers

In [ ]:
qa = [
    {
        "q": "What are the four components of MLflow and what does each do?",
        "a": """
1. MLflow Tracking: logs parameters, metrics, and artifacts for each training run.
   You call mlflow.log_param(), log_metric(), log_artifact() in your training code.
   All runs are stored in a database (local file or remote server) and viewable via MLflow UI.

2. MLflow Models: a standard format for packaging ML models from any framework.
   A 'model' directory contains: MLmodel file (metadata), model files, conda.yaml (environment).
   Supports 'flavors': sklearn, pytorch, tensorflow, pyfunc (generic Python function).

3. MLflow Model Registry: version control + lifecycle management for production models.
   Models go through stages: None → Staging → Production → Archived.
   Provides auditability: who promoted a model, when, why.

4. MLflow Projects: reproducible packaging of ML code as a directory with an MLproject file.
   Defines entry points, parameters, and dependencies. Run with: mlflow run .
   Less commonly used in practice (Docker/Poetry often preferred for reproducibility).
        """
    },
    {
        "q": "MLflow vs Weights & Biases — when would you choose each?",
        "a": """
Choose MLflow when:
- You need a free, fully self-hosted solution (run your own server)
- Your org requires all data on-premises (data privacy, compliance)
- You want a model registry with production deployment integration
- You're using Databricks (MLflow is deeply integrated)
- You need multi-framework support (sklearn, PyTorch, TF, etc.)

Choose Weights & Biases when:
- You need beautiful, shareable dashboards for stakeholders
- Your team is remote and collaboration/sharing is priority
- You're running lots of hyperparameter sweeps (W&B Sweeps is excellent)
- You want rich media logging: images, audio, video, 3D objects, custom plots
- You're doing DL research where visualizing training dynamics matters

Both can track parameters/metrics/artifacts. The difference:
MLflow = enterprise/self-hosted/model-registry-focused
W&B = team-collaboration/rich-media/sweep-focused
        """
    },
    {
        "q": "What is the MLflow Model Registry and why is it important?",
        "a": """
The Model Registry is a centralized store for managing production ML models.

Before registry: teams deployed models by manually copying weight files, no versioning,
no auditability, no easy rollback. "Which model is in production?" was answered by
hunting through S3 buckets or asking colleagues.

The registry provides:
1. Versioning: each registered model has an auto-incrementing version number
2. Lifecycle stages: None → Staging → Production → Archived
   - Staging = tested, ready for review
   - Production = currently serving traffic  
   - Archived = no longer active, kept for history
3. Annotations: add descriptions, tags, and aliases to versions
4. Lineage: each version links to its training run — full provenance
5. Serving code uses aliases ("Production") not version numbers:
   mlflow.pyfunc.load_model("models:/churn_model/Production")
   When you promote v2 to Production, serving code automatically uses the new model.
        """
    },
    {
        "q": "How does mlflow.autolog() work and what are its limitations?",
        "a": """
mlflow.autolog() (or mlflow.sklearn.autolog()) patches the training framework at the
Python level. It wraps fit(), __init__(), and other methods to intercept calls and
automatically log parameters, metrics, and models.

How it works:
1. Patches the framework's training methods before fit() is called
2. Extracts hyperparameters from __init__() arguments
3. Extracts metrics from model attributes (e.g., best_score_, oob_score_)
4. Saves the model artifact after fit() completes

Limitations:
1. Not all frameworks supported (custom models need manual logging)
2. Cross-validation metrics may be duplicated (run-level + CV-level)
3. Can be too verbose — logs every intermediate metric
4. Nested runs: some operations create nested runs (CV, GridSearchCV)
5. In Jupyter notebooks: repeated cell execution creates duplicate runs
   Fix: call mlflow.end_run() before re-running, or use `with mlflow.start_run():`
6. Cannot capture custom metrics you compute yourself
   → For custom metrics, combine autolog() with manual log_metric() calls
        """
    },
    {
        "q": "How would you set up MLflow for a team of 10 ML engineers?",
        "a": """
For a team, local filesystem tracking is insufficient — you need a shared tracking server.

Production setup:
1. MLflow Tracking Server:
   mlflow server \\
     --backend-store-uri postgresql://user:pw@host/mlflow  # metadata DB
     --default-artifact-root s3://company-bucket/mlflow   # artifact storage
     --host 0.0.0.0 --port 5000

2. Each engineer sets:
   export MLFLOW_TRACKING_URI=http://mlflow.internal:5000
   export AWS_PROFILE=mlflow_role  # for S3 access

3. Authentication: add nginx reverse proxy with basic auth, or use
   Databricks MLflow (managed, with SSO and RBAC built-in)

4. Model Registry: automatically shared since it's on the same server

5. Naming conventions: agree on experiment naming (team_project_version)
   and mandatory tags (author, ticket, dataset_version)

Managed alternatives: Databricks MLflow, Azure ML (MLflow compatible),
AWS SageMaker MLflow integration
        """
    },
    {
        "q": "How do MLflow parameters, metrics, and artifacts differ?",
        "a": """
Parameters (log_param / log_params):
- Inputs to your training run: hyperparameters, data paths, configurations
- String key-value pairs (values stored as strings)
- Logged ONCE per run (immutable after logging)
- Examples: learning_rate=0.001, n_layers=3, dataset_version=v2.1

Metrics (log_metric / log_metrics):
- Numeric outputs you measure during/after training
- Float values; can be logged MULTIPLE TIMES with a step counter
- MLflow stores the full history (for training curves)
- Examples: accuracy, loss, val_auc, latency_ms

Artifacts (log_artifact / log_artifacts):
- Files: model weights, plots, confusion matrices, data samples, configs
- Stored in artifact storage (local filesystem, S3, GCS, Azure Blob)
- Any file type: .pkl, .pt, .png, .csv, .json
- Accessed via mlflow.artifacts.download_artifacts()

Rule of thumb:
  param = 'what I set before training'
  metric = 'what I measured during/after training'
  artifact = 'files I want to keep from this run'
        """
    },
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 70)
    print()

## Summary

| Feature | What It Does | Key API |
|---------|-------------|---------|
| Tracking | Log params/metrics/artifacts | `mlflow.log_param()`, `log_metric()`, `log_artifact()` |
| Experiments | Group related runs | `mlflow.set_experiment()` |
| Autolog | Auto-capture from frameworks | `mlflow.sklearn.autolog()` |
| Model logging | Save/load models | `mlflow.sklearn.log_model()`, `load_model()` |
| Registry | Version + lifecycle mgmt | `mlflow.register_model()` |
| Serving | REST API from model | `mlflow models serve` |
| Search | Query across runs | `mlflow.search_runs()` |

### Next Steps

1. **Set up a local MLflow server**: `mlflow server --backend-store-uri sqlite:///mlflow.db`
2. **MLflow + scikit-learn tutorial**: [https://mlflow.org/docs/latest/tutorials-and-examples/tutorial.html](https://mlflow.org/docs/latest/tutorials-and-examples/tutorial.html)
3. **MLflow + PyTorch**: [https://mlflow.org/docs/latest/python_api/mlflow.pytorch.html](https://mlflow.org/docs/latest/python_api/mlflow.pytorch.html)
4. **Compare with DVC**: DVC is better for data versioning; MLflow for experiment tracking
5. **Next**: Learn Weights & Biases for team collaboration and richer visualizations